In [ ]:
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning, MarkupResemblesLocatorWarning
import requests
from urllib.parse import urlparse, urljoin
import random
import warnings
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import os

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

df = pd.read_csv('../../datasets/alexa-top-1m/top-1m-resolved.csv', index_col=0)

df.shape

(714866, 1)

In [3]:
df = df.sample(frac=1, ignore_index=True)

df

,url
0,https://cambio.bo/
1,https://visuallightbox.com/
2,https://wopart.eu/
3,https://osteomed-clinic.ru/
4,https://www.lanac.com.br/
...,...
714861,https://picpa.org/
714862,https://www.teamskeet.com/series/bffs
714863,https://eyesonisles.com/
714864,https://www.aulro.com/


In [ ]:
def extract_links(url, max_links=7):
  """
  Using Beautifulsoup, it web scapes urls from the resolved url web page
  """
  try:
    response = requests.get(
      url,
      timeout=10,
      headers={"User-Agent": "Mozilla/5.0"},
      allow_redirects=True
    )
    
    if response.status_code != 200:
      return set()
        
    if 'text/html' not in response.headers.get('Content-Type', ''):
      return set()

    soup = BeautifulSoup(response.text, "html.parser")
    links = set()
    
    for a in soup.find_all("a", href=True):
      href = a["href"]
      # Only keep links that belong to the same domain
      if href.startswith("http"):
        if urlparse(href).netloc == urlparse(url).netloc:
          links.add(href)
      # Convert relative paths to absolute
      elif href.startswith(("/", "#")):
        links.add(urljoin(url, href))
    links.discard(None)
    if len(links) > max_links:
      links = set(random.sample(list(links), max_links))
    return links
  except:
    return set()

def extract_links_with_retry(url, max_retries=3):
    """
    Extract links with exponential backoff on failure
    """
    for attempt in range(max_retries):
        try:
            return extract_links(url)
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:
                wait_time = (2 ** attempt) * 1
                time.sleep(wait_time)
            elif e.response.status_code in [403, 503]:
                return set()
            else:
                raise
        except requests.exceptions.Timeout:
            time.sleep(2 ** attempt)
    
    return set()

In [5]:
extract_links_with_retry('https://www.youtube.com/')

{'https://www.youtube.com/',
 'https://www.youtube.com/about/',
 'https://www.youtube.com/about/copyright/',
 'https://www.youtube.com/creators/',
 'https://www.youtube.com/howyoutubeworks?utm_campaign=ytgen&utm_source=ythp&utm_medium=LeftNav&utm_content=txt&u=https%3A%2F%2Fwww.youtube.com%2Fhowyoutubeworks%3Futm_source%3Dythp%26utm_medium%3DLeftNav%26utm_campaign%3Dytgen',
 'https://www.youtube.com/new',
 'https://www.youtube.com/t/terms'}

In [ ]:
def process_chunk(chunk, max_workers=100):
  """
  Process a single chunk of URLs concurrently with threads.
  """
  enriched_urls = 0
  results = [None] * len(chunk)
  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(extract_links, chunk.iloc[i]): i for i in range(len(chunk))}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving chunk"):
      idx = futures[future]
      result = future.result()
      results[idx] = result
      enriched_urls += len(result) > 0
  print(f'{enriched_urls} url enriched')
  return results

In [7]:
def enrich_urls(df, chunks_folder, chunk_size=50000, sub_chunk_size=None, max_workers=100):
    """
    Enrich all URLs in a DataFrame in chunks with threading.
    Returns the DataFrame with a new column 'enriched_url'.
    """
    processed_chunks = []
    for file in os.listdir(chunks_folder):
        if file.endswith('.csv'):
            processed_chunks.append(int(file.split('.')[0].split('_')[1]))
    chunks = [df.iloc[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

    for i, chunk in enumerate(chunks):
        if i in processed_chunks:
            continue
        print(f"Processing sub chunk {i + 1}/{len(chunks)} ({len(chunk)} URL)")
        chunk_results = []
        if sub_chunk_size:
            sub_chunks = [chunk.iloc[i:i+sub_chunk_size] for i in range(0, len(chunk), sub_chunk_size)]
            for sub_chunk in sub_chunks:
                sub_chunk_results = process_chunk(sub_chunk['url'], max_workers=max_workers)
                chunk_results.extend(sub_chunk_results)
        else:
            chunk_results = process_chunk(chunk['url'], max_workers=max_workers)

        chunk['enriched_url'] = chunk_results
        chunk.to_csv(f'{chunks_folder}/chunk_{i}.csv')

In [8]:
chunk_size = len(df) // 50
sub_chunk_size = chunk_size // 3
chunks_folder = '../../datasets/alexa-top-1m/chunks/enriched'

enrich_urls(df, chunks_folder=chunks_folder, chunk_size=chunk_size, sub_chunk_size=sub_chunk_size, max_workers=50)


Processing sub chunk 32/51 (14297 URL)


Resolving chunk: 100%|██████████| 4765/4765 [09:53<00:00,  8.03it/s]


3094 url enriched


Resolving chunk: 100%|██████████| 4765/4765 [08:03<00:00,  9.86it/s]


3187 url enriched


...

3197 url enriched


Resolving chunk: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]


1 url enriched
Processing sub chunk 51/51 (16 URL)


Resolving chunk: 100%|██████████| 16/16 [00:10<00:00,  1.46it/s]

12 url enriched
